MMM E2 is a classifier predicting alpha breakouts from companies within the NDXT. Alpha is calculated between company and NDXT, not S&P500. 3 classes (labels) are as follows:
-

Leaders (Class 1): Predicted statistical outperformance by 1 sd

Laggards (Class -1): Predicted statistical underperformance by 1 sd

Neutral (Class 0): Stocks expected to track the index normally, within 1 sd

#Downloads and imports (~30s)

In [ ]:
#!pip install yfinance

In [ ]:
#! pip install cudf-cu11 --extra-index-url=https://pypi.ngc.nvidia.com
#! pip install cuml-cu11 --extra-index-url=https://pypi.ngc.nvidia.com
# The exact version (e.g., cu11) might change, check the official RAPIDS installation guide for the latest command.

In [ ]:
try:
  %load_ext cuml.accel
except Exception as e:
  print(e)

No module named 'cuml'


In [ ]:
# download TA-Lib
!wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
#!ls
!tar xvzf ta-lib-0.4.0-src.tar.gz
#!ls
import os
os.chdir('ta-lib') # Can't use !cd in co-lab
!./configure --prefix=/usr
!make
!make install
# wait ~ 30s
os.chdir('../')
#!ls
!pip install -q TA-Lib

ta-lib/
ta-lib/config.sub
ta-lib/aclocal.m4
ta-lib/CHANGELOG.TXT
ta-lib/include/
ta-lib/include/ta_abstract.h
ta-lib/include/ta_func.h
ta-lib/include/ta_common.h
ta-lib/include/ta_config.h.in
ta-lib/include/Makefile.am
ta-lib/include/ta_libc.h
ta-lib/include/ta_defs.h
ta-lib/missing
ta-lib/ta-lib.spec.in
ta-lib/config.guess
ta-lib/Makefile.in
ta-lib/ta-lib.dpkg.in
ta-lib/Makefile.am
ta-lib/autogen.sh
ta-lib/install-sh
ta-lib/configure
ta-lib/depcomp
ta-lib/HISTORY.TXT
ta-lib/configure.in
ta-lib/autom4te.cache/
ta-lib/autom4te.cache/output.0
ta-lib/autom4te.cache/requests
ta-lib/autom4te.cache/output.1
ta-lib/autom4te.cache/traces.0
ta-lib/autom4te.cache/traces.1
ta-lib/ltmain.sh
ta-lib/ta-lib-config.in
ta-lib/src/
ta-lib/src/ta_func/
ta-lib/src/ta_func/ta_MACDFIX.c
ta-lib/src/ta_func/ta_CDLPIERCING.c
ta-lib/src/ta_func/ta_DIV.c
ta-lib/src/ta_func/ta_ROCR100.c
ta-lib/src/ta_func/ta_ADXR.c
ta-lib/src/ta_func/ta_MAVP.c
ta-lib/src/ta_func/ta_CDLCLOSINGMARUBOZU.c
ta-lib/src/ta_func/ta_COSH.

In [ ]:
import talib as ta
import random
import time
import datetime as dt
from datetime import datetime, timedelta
import numpy as np
import yfinance as yf
import pandas as pd
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning) #ignores "Dataframe is highly fragmented" warning.
from sklearn import preprocessing
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (AdaBoostClassifier, RandomForestClassifier,
                              StackingClassifier, HistGradientBoostingClassifier,
                              ExtraTreesClassifier, BaggingClassifier, VotingClassifier, GradientBoostingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from matplotlib import pyplot as plt
import math
import lightgbm as lgb
from dateutil.relativedelta import relativedelta
import statistics as stat
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def change_days(date_str, days):
    import datetime as dt
    date = dt.datetime.strptime(date_str, "%Y-%m-%d")

    new_date = date + timedelta(days=days)

    return new_date.strftime("%Y-%m-%d")

def change_months(date_str, monthss):
  import datetime as dt
  date_obj = datetime.strptime(date_str, "%Y-%m-%d")

  new_date = date_obj + relativedelta(months=monthss)

  return new_date.strftime("%Y-%m-%d")

#Functions (~10s)


In [ ]:
def spread_zscore(df, df2): #finds the spread's z_score
  lookback = 22
  df3 = pd.DataFrame()
  df3['sprd'] = df - df2
  df3['sprd_mean'] = df3['sprd'].rolling(window=lookback).mean()
  df3['sprd_std'] = df3['sprd'].rolling(window=lookback).std()
  df3['sprd_zscore'] = ( df3['sprd'] - df3['sprd_mean'] ) / df3['sprd_std']
  return df3['sprd_zscore']

def technical_data_1(df, lookback = 14): #This processes stock data the first time, returns necessary indicators for the second round of processing

  for i in range(1, 60):
    df[f"Close -{i}d"] = df["Close"].shift(i)
    df[f"High -{i}d"] = df["High"].shift(i)
    df[f"Low -{i}d"] = df["Low"].shift(i)

  #🔴Price slope and ROC is similar, may be redundant
  df['RSI'] = ta.RSI(df['Close'], lookback)
  df['NATR'] = ta.NATR(df['High'], df['Low'], df['Close'], lookback)
  df['BB_U'], df['BB_M'], df['BB_L'] = ta.BBANDS(df['Close'], lookback)
  df['BB_W'] = ( ( df['BB_U'] - df['BB_L'] ) / df['BB_M'] ) * 100
  df['ROC'] = ta.ROC(df['Close'], lookback)
  df['Volume_RSI'] = ta.RSI(df['Volume'], lookback)
  df['Price_slope'] = ta.LINEARREG_SLOPE(df['Close'], lookback)
  df['K%'], df['D%'] = ta.STOCH(df['High'], df['Low'], df['Close'], fastk_period=5, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)

  for i in range(1,60):
    df = df.drop(columns=[f'Close -{i}d', f'High -{i}d', f'Low -{i}d'])

  return df

def technical_data_2(df, df2): #Replaces old features with features that capture divergence between df and df2 (security and NDXT)
  if df.shape != df2.shape:
    return 'ERROR, DF SHAPES ARE DIFFERENT.'
  df3 = pd.DataFrame()

  df3['RSI_sprd'] = df["RSI"] - df2['RSI']
  df3['ROC_sprd_zscore'] = spread_zscore(df['ROC'], df2['ROC'])
  df3['NATR_sprd_zscore'] = spread_zscore(df['NATR'], df2['NATR'])
  df3['BBW_sprd_zscore'] = spread_zscore(df['BB_W'], df2['BB_W'])
  df3['Volume-RSI_sprd_zscore'] = spread_zscore(df['Volume_RSI'], df2['Volume_RSI'])
  df3['Price-slope_sprd_zscore'] = spread_zscore(df['Price_slope'], df2['Price_slope'])
  df3['K%_sprd_zscore'] = spread_zscore(df['K%'], df2['K%'])

  return df3

def complete_technical_data(df, df2, lookback): #lookback is lookback period for calculating technical indicators
  df = technical_data_1(df, lookback)
  df2 = technical_data_1(df2)
  df3 = technical_data_2(df, df2)
  df3['Close'] = df['Close'] # NEEDED FOR CALCULATING LABEL
  return df3


def data(stock, startdate, enddate, beta_lookback, risk_free_rate, z_score_threshold, indicators_lookback, dropna = True): #The Complete data retrieval function
  df = pd.read_excel(f'/content/drive/MyDrive/SP/NDXT_data/{stock.upper()}.xlsx', index_col = 0).loc[startdate : enddate]
  NDXT = pd.read_excel(f'/content/drive/MyDrive/SP/NDXT_data/^NDXT.xlsx', index_col = 0).loc[startdate : enddate]

  df = complete_technical_data(df, NDXT, indicators_lookback)

  #Calculating daily returns for Beta:
  df['NDXT_Close'] = NDXT['Close']
  df['NDXT_daily_returns'] = df['NDXT_Close'].pct_change()
  df['Daily_returns'] = df['Close'].pct_change()

  #Calculating Beta:
  df['Cov'] = df['Daily_returns'].rolling(window=beta_lookback).cov(df['NDXT_daily_returns'])
  df['Var'] = df['NDXT_daily_returns'].rolling(window=beta_lookback).var()
  df["Beta"] = df['Cov']/df['Var']

  #Calculating returns:
  df['Returns'] = (df['Close'].shift(-22) - df['Close']) / df['Close']
  df['NDXT_returns'] = (df['NDXT_Close'].shift(-22) - df['NDXT_Close']) / df['NDXT_Close']

  #Calculating alpha:
  df['Market_premium'] = df['NDXT_returns'] - risk_free_rate
  df['Expected_returns'] = risk_free_rate + ( df['Market_premium'] * df['Beta'] )
  df['Alpha'] = df['Returns'] - df['Expected_returns']

  #Calculating z-scored alpha
  df['Alpha_mean'] = df['Alpha'].rolling(window=50).mean().shift(22) #The shift makes it so the mean alpha tday is the mean from the last month, not the next month
  df['Alpha_std'] = df['Alpha'].rolling(window=50).std().shift(22) #Zscored alpha should use mean and sd from a month ago
  df['Alpha_zscore'] = ( df['Alpha'] - df['Alpha_mean'] ) / df['Alpha_std']

  #Creating addiional features
  df['Adjusted_Alpha_zscore'] = df['Alpha_zscore'].shift(23)
  k = 22
  for i in range(1,4):
    df[f'Adjusted_Alpha_z_score_slope_{i}'] = ta.LINEARREG_SLOPE(df['Adjusted_Alpha_zscore'], timeperiod = int(k/i))

  #Creating Labels
  df['Label'] = np.where(df['Alpha_zscore'] > z_score_threshold, 1,
                        np.where(df['Alpha_zscore'] < (-1 * z_score_threshold), -1,
                                 np.where((df['Alpha_zscore'] > -z_score_threshold) & (df['Alpha_zscore'] < z_score_threshold), 0, 911)))

  df = df.drop(columns = ['Close', 'NDXT_Close', 'NDXT_daily_returns', 'Daily_returns', 'Cov', 'Var', 'Expected_returns', 'Returns', 'NDXT_returns', 'Market_premium','Alpha', 'Alpha_mean', 'Alpha_std', 'Alpha_zscore'])

  if dropna:
    df = df.dropna()

  return df


#📌 pred_data function requires proof reading to ensure data created is accurate.
def pred_data(stock, date, beta_lookback, risk_free_rate, z_score_threshold, indicators_lookback):
  X_pred = data(stock, change_months(date, -13), date, beta_lookback, risk_free_rate, z_score_threshold, indicators_lookback, dropna = False).tail(1).drop(columns = ['Label'])
  return X_pred

#tie-breaking
def tie_break(l, epsilon=0.001):
  seen = {}
  for i, x in enumerate(l):
      if x in seen:
          count = seen[x]
          l[i] += epsilon * count
          seen[x] += 1
      else:
          seen[x] = 1
  return l

def technical_model(stock, date):
  k=0#####

def backtest_technical_model(stock, date):
  df = data(stock, change_months(date, -150), date, 126, 0.003674809, 1, 15, True).iloc[:-22]
  y = df.pop('Label')
  X = df

  X_pred = pred_data(stock, date, 126, 0.003674809, 1, 15)

  clf.fit(X, y)
  pred = clf.predict(X_pred)
  proba = clf.predict_proba(X_pred)

  return pred, proba[0][0], proba[0][2], proba[0][2]-proba[0][0] #Verdict, bear proba, bull proba, weighted bull_proba

In [ ]:
#Different backtest methods, chandelier only does 1 month :)

def backtest_exit_rules(stock, date, stoploss = 0.1 ,  action = 'long', timeframe = 'month', days = 0):#function will return how much the stock price changed in the coming month with exit rules
  if timeframe == 'month':
    enddate = change_months(change_days(date, 1), 1)
  if timeframe == 'week':
    enddate = change_days(change_days(date, 1), 7)
  if timeframe == 'custom':
    enddate = change_days(change_days(date, 1), days)
  print('\n',f"[------=====◼️◼️◼️◼️◼️ Backtesting {action} {stock} [{date} --> {enddate}] ◼️◼️◼️◼️◼️=====--------]")

  initial_price = yf.Ticker(stock).history(start = change_days(date, -5), end = date).tail(1)
  new_price = yf.Ticker(stock).history(start = date, end = enddate).tail(1)

  #print(order)
  pct_change = (float(new_price["Close"].iloc[0])-float(initial_price["Close"].iloc[0]))/float(initial_price["Close"].iloc[0])

  print(f' initial close: {initial_price["Close"].iloc[0]}, new close: {new_price["Close"].iloc[0]}')

  if action == 'long':
    order = yf.Ticker(stock).history(start=date, end=enddate)['Low']
    stop_loss_price = float(initial_price["Close"].iloc[0])*(1-stoploss)
    for i in order:
      if i < stop_loss_price:
        print(f"[---------🔴 Long {stock} price decreased by over {stoploss*100}%, stop loss triggered 🔴 ---------]")
        return -stoploss

    print(f"[--------- {stock} Percentage Gain: {pct_change} ---------]")

    return pct_change

  elif action =='short':
    order = yf.Ticker(stock).history(start=date, end=enddate)['High']
    stop_loss_price = float(initial_price['Close'].iloc[0])*(1+stoploss)
    for i in order:
      if i > stop_loss_price:

        print(f"[---------🔴 Short {stock} price increased by over {stoploss*100}%, stop loss triggered 🔴---------]")
        return -stoploss

    print(f"[--------- {stock} Percentage Gain: {pct_change*-1} ---------]")
    return pct_change*-1



  else:
    print(' PLEASE ENTER ACTION (LONG/SHORT) ')
  return pct_change

def Chandelier_exit(stock, date, multiplier = 4, action = 'long'):

  print_info = lambda stock, action, i, loss, pct_loss, starting_price, current_price, exit_price: print(f''' {"= "*20}{'\n'} [{stock.upper()}, {action}] {'\n'} Exited early on {i}th day, {'\n'} loss: {round(loss, 5)}, percentage loss: {round(pct_loss, 5)} {'\n'*2} Starting Price: {starting_price} {'\n'} Price Settled: {current_price}{'\n'} Stoploss Price: {exit_price} ''')

  exit_prices = []

  starting_price = yf.Ticker(stock).history(start = change_days(date, -10), end = change_days(date, 1))['Close'].tail(1).iloc[0] #Get starting price for calculating profits and losses. End date is changed because yfinance dates are exclusive, so by shifting it by one day, it includes data for date

  for i in range(1,23):
    df = yf.Ticker(stock).history(start = change_days(date, -45), end = date) #Leaves room for ATR to be calculated

    df['ATR'] = ta.ATR(df['High'], df['Low'], df['Close'], timeperiod = 22)
    df = df[-22:]
    ATR = df['ATR'].tail(1).iloc[0] #Gets "Today's" ATR

    high_or_low = df['High'].max() if action == 'long' else df['Low'].min() #Highest high is used when longing, Lowest low is used when shorting when calculating exit price
    operation = '-' if action == 'long' else '+' # When longing, exit price is highest high - ATR*K; When shorting, exit price is lowest low + ATR*K

    current_price = df['Low' if action == 'long' else 'High'].tail(1).iloc[0] #Long and shorts looks at Low/High when determining whether or not to exit
    exit_price = eval(f'high_or_low{operation}(ATR*multiplier)')
    exit_prices.append(exit_price)

    date = change_days(date, 1)

    if action == 'long':
      exit_price = max(exit_prices) #Ratchet rule, not a standard ratchet rule, but one that doesnt vary with volatility

      if current_price < exit_price:
        loss = abs(starting_price - exit_price)
        pct_loss = loss/starting_price
        print_info(stock, action, i, loss, pct_loss, starting_price, current_price, exit_price)
        return pct_loss

    else:
      exit_price = min(exit_prices) #Aforementioned ratchet rule

      if current_price > exit_price:
        loss = abs(exit_price - starting_price)
        pct_loss = loss/starting_price
        print_info(stock, action, i, loss, pct_loss, starting_price, current_price, exit_price)
        return pct_loss

def backtest_chandelier_exit(stock, date, multiplier, action = 'long'):
  pct_loss = Chandelier_exit(stock, date, multiplier=multiplier, action=action)

  if pct_loss:
    return pct_loss*-1

  return backtest_exit_rules(stock, date, stoploss=100, action = action, timeframe = 'month')

In [ ]:
stock = 'nvda'
enddate = '2026-05-11'
startdate = change_months(enddate, -150)
beta_lookback = int(252/2)
risk_free_rate = 0.003674809
z_score_threshold = 1
timeperiod = 15 #Days technical indicators lookback in calculation

df = data(stock, startdate, enddate, beta_lookback, risk_free_rate, z_score_threshold, timeperiod, dropna = True)

y = df.pop('Label')
X = df

In [ ]:
#clf = RandomForestClassifier(random_state=0, class_weight='balanced', max_depth=3) #Vote ensemble might do better.
test_size = 400

X_test = X[-test_size:]
y_test = y[-test_size:]
X_train = X[:-test_size]
y_train = y[:-test_size]

X_train = pd.concat([X_train, y_train], axis=1)
X_train = X_train.sample(frac=1, random_state=0)
y_train = X_train.pop('Label')

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
cr = classification_report(y_test, y_pred)

print(X_train.shape, X_test.shape)
print(f'Training Accuracy: {clf.score(X_train, y_train)}') # to check for overfitting, found that slightly underfitted RFs perform best
print(cm, '\n', cr)

(2490, 12) (400, 12)
Training Accuracy: 0.5172690763052209
[[ 45  14  30   0]
 [ 52  50 111   0]
 [ 30   7  39   0]
 [  3  11   8   0]] 
               precision    recall  f1-score   support

          -1       0.35      0.51      0.41        89
           0       0.61      0.23      0.34       213
           1       0.21      0.51      0.30        76
         911       0.00      0.00      0.00        22

    accuracy                           0.34       400
   macro avg       0.29      0.31      0.26       400
weighted avg       0.44      0.34      0.33       400



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
def print_class_pct(series):
  x1 = (series == -1).sum()
  x2 = (series == 0).sum()
  x3 = (series == 1).sum()
  x_sum = x1+x2+x3
  print(f'''
  Class -1: {int((x1/x_sum)*100)}%
  Class 0: {int((x2/x_sum)*100)}%
  Class 1: {int((x3/x_sum)*100)}%
  ''')

print_class_pct(y_train)
print_class_pct(y_test)


  Class -1: 25%
  Class 0: 47%
  Class 1: 26%
  

  Class -1: 23%
  Class 0: 56%
  Class 1: 20%
  


#Loading Algorithms

In [ ]:
ADARF = AdaBoostClassifier(estimator = RandomForestClassifier(random_state=1, class_weight='balanced'),random_state=1)
RF = RandomForestClassifier(random_state = 1, class_weight='balanced')
GB = HistGradientBoostingClassifier(max_iter = 2000, random_state = 1)
ET = ExtraTreesClassifier(random_state = 1, class_weight='balanced')
BGGB = BaggingClassifier(estimator = RandomForestClassifier(random_state = 1), random_state =1)
#lgb classifier not working
LGBM = lgb.LGBMClassifier(verbose=-1, random_state=1, class_weight='balanced')

clf = VotingClassifier(estimators=[
            ('ADARF', ADARF), ('RF', RF),('LGBM',LGBM),('ET', ET)], voting='soft')

print("✅ Done loading the model ✅")
#vote_score = cross_val_score(vote, X,y,cv=5)

#On SPGI: Cross validation score:
#[0.90106007 0.89399293 0.90106007 0.89399293 0.87588652]

✅ Done loading the model ✅


#Testing

In [ ]:
stocks = [
    "NVDA", "GOOGL", "GOOG", "AAPL", "MSFT", "AMZN", "AVGO", "META", "TSLA", "WMT",
    "AMD", "ASML", "MU", "COST", "INTC", "NFLX", "CSCO", "PLTR", "LRCX", "AMAT",
    "KLAC", "TXN", "ARM", "LIN", "PEP", "TMUS", "ADI", "AMGN", "ISRG", "SHOP",
    "GILD", "QCOM", "APP", "PANW", "MRVL", "BKNG", "PDD", "WDC", "HON", "STX",
    "CRWD", "CEG", "SBUX", "INTU", "VRTX", "ADBE", "CMCSA", "MAR", "SNPS", "MELI",
    "CDNS", "ABNB", "CSX", "MPWR", "ADP", "ORLY", "DASH", "MNST", "REGN", "MDLZ",
    "AEP", "ROST", "CTAS", "BKR", "WBD", "PCAR", "FTNT", "NXP", "MSTR", "FANG",
    "FAST", "EA", "ADSK", "FER", "XEL", "MCHP", "EXC", "ODFL", "DDOG", "PYPL",
    "IDXX", "CCEP", "ALNY", "KDP", "TRI", "TTWO", "ROP", "PAYX", "AXON", "CPRT",
    "GEHC", "WDAY", "INSM", "CTSH", "KHC", "CHTR", "DXCM", "VRSK", "ZS", "TEAM",
    "CSGP"]
len(stocks)

101

In [ ]:
def random_date(start, end): #years only, im lazy
  year = str(random.randint(start, end))
  month = str(random.randint(1, 12))
  day = str(random.randint(1, 28))
  fx = lambda X : "0"+X if len(list(X)) == 1 else X
  month = fx(month)
  day = fx(day)
  return(year+"-"+month+"-"+day)


In [ ]:
top_k = 5
overall_change = []

for i in range(5):
  try:
    date = random_date(2020, 2025)

    bullarities = []
    bullest_stocks = []
    bearest_stocks = []
    long_returns = []
    short_returns = []

    for stock in stocks:
      try:
        verdict, bull_proba, bear_proba, bullarity = backtest_technical_model(stock, date)
        bullarities.append(bullarity)
        print(f'🔹{stock}: {verdict}, {bullarity}')
      except Exception as e:
        print(f'🔺{stock} Errored: {e}')
        bullarities.append(0)

    bullarities = tie_break(bullarities)
    bullarities_sorted = sorted(bullarities)

    lowest = bullarities_sorted[:top_k]
    highest = bullarities_sorted[-top_k:]

    for i in range(top_k):
      bullest_stocks.append(stocks[bullarities.index(highest[i])])
      bearest_stocks.append(stocks[bullarities.index(lowest[i])])

    print(f''' 🔹🔹🔹{date}🔹🔹🔹
    Bullest Stocks: {bullest_stocks}
    Bearest Stocks: {bearest_stocks}
    🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹
    ''')

    for stock in bullest_stocks:
      returns  = backtest_exit_rules(stock, date, stoploss = 0.15, action = 'long')
      long_returns.append(returns)

    for stock in bearest_stocks:
      returns = backtest_exit_rules(stock, date, stoploss = 0.15, action = 'short')
      short_returns.append(returns)

    long_returns = sum(long_returns)/len(long_returns)
    short_returns = sum(short_returns)/len(short_returns)

    monthly_change = long_returns + short_returns
    overall_change.append(monthly_change)

    print(f'''
    🔹🔹🔹{date}🔹🔹🔹
    🔹{monthly_change}🔹

    ======================================================================================
    ''')
  except Exception as e:
    print(f'🟥🟥🟥 Error Occurred: {e}')

print(overall_change)
k=1
for i in overall_change:
  k = k*(i+1)

k

🔹NVDA: [-1], -0.5851357354795317
🔹GOOGL: [1], 0.3639462016441788
🔹GOOG: [0], 0.11970704482751418
🔹AAPL: [0], -0.23445256099344727
🔹MSFT: [-1], -0.43358296548662995
🔹AMZN: [0], -0.0014222954635003116
🔺AVGO Errored: 'str' object does not support item assignment
🔺META Errored: 'str' object does not support item assignment
🔺TSLA Errored: 'str' object does not support item assignment
🔹WMT: [1], 0.21432655479336044
🔹AMD: [-1], -0.4898391227514259
🔹ASML: [0], 0.04652432275073398
🔹MU: [0], -0.03633538527700422
🔹COST: [0], -0.10915778621875405
🔹INTC: [1], 0.27460867817128454
🔹NFLX: [0], -0.008609688518574521
🔹CSCO: [1], 0.38591932094689146
🔺PLTR Errored: 'str' object does not support item assignment
🔹LRCX: [0], 0.029272430160515145
🔹AMAT: [-1], -0.40948843092821374
🔹KLAC: [0], -0.19453652192097667
🔹TXN: [0], 0.04793356087117723
🔺ARM Errored: 'str' object does not support item assignment
🔹LIN: [-1], -0.4216527768788796
🔹PEP: [0], -0.03401046724790063
🔹TMUS: [0], -0.07291319799441776
🔹ADI: [0], -

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


🔺FER Errored: Input X contains infinity or a value too large for dtype('float64').
🔹XEL: [-1], -0.09827767555942829
🔹MCHP: [0], -0.1473275666051391
🔹EXC: [0], -0.05758590866511748
🔹ODFL: [0], -0.10088627199492475
🔺DDOG Errored: 'str' object does not support item assignment
🔺PYPL Errored: 'str' object does not support item assignment
🔹IDXX: [0], 0.19919209138105962
🔹CCEP: [-1], -0.4446826139539982
🔹ALNY: [0], -0.10071419195275794
🔹KDP: [0], -0.024637288543927188
🔹TRI: [0], -0.1130697084665608
🔹TTWO: [0], 0.05172058120631365
🔹ROP: [0], -0.27231583676856214
🔹PAYX: [0], -0.21586825287856143
🔹AXON: [0], -0.026681200219075635
🔹CPRT: [0], 0.20469456579233958
🔺GEHC Errored: 'str' object does not support item assignment
🔺WDAY Errored: 'str' object does not support item assignment
🔹INSM: [0], -0.11573888012389807
🔹CTSH: [0], -0.0570860947551835
🔺KHC Errored: 'str' object does not support item assignment
🔹CHTR: [-1], -0.18808463978460388
🔹DXCM: [-1], -0.47558760595871097
🔹VRSK: [1], 0.05378165723

0.8442693920598896

In [ ]:
backtest_technical_model('^spx', date)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/SP/NDXT_data/^SPX.xlsx'

In [ ]:
bearest_stocks, bullest_stocks

(['aapl'], ['msft'])

In [ ]:
#VOLATILITY TEST, MARKET NEUTRAL +  STOPLOSS ON VEY VOLATILE STONKS
stoploss = 0.04

stocks = ['TSLA', 'NVDA', 'AMD', 'ENPH', 'SEDG', 'CCL', 'NCLH', 'RCL', 'MGM', 'Wynn', 'MRNA', 'DXCM', 'AFRM', 'PYPL', 'APA', 'DVN', 'OXY', 'ETSY', 'UBER', 'LYFT', 'POOL']
changes = []

for i in range(150):
  date = random_date(2022, 2025)
  choices = random.choices(stocks, k=10)
  long_changes = []
  short_changes = []

  for i in range(5):
    long_changes.append(backtest_exit_rules(choices[i], date, stoploss = stoploss, action = 'long'))
    short_changes.append(backtest_exit_rules(choices[-1*(i+1)], date, stoploss = stoploss, action = 'short'))

  returns = (sum(long_changes)/len(long_changes)) + (sum(short_changes)/len(short_changes))
  changes.append(returns)
  print("\n"*5, date, returns, "\n"*5)



NameError: name 'backtest_exit_rules' is not defined

In [ ]:
k = 1
for i in changes:
  k = k*(i+1)

round(k, 3)

1.942

In [ ]:
############# NEWEST BACKTEST FUCNTION 2026-04-15

date = '2020-04-01'
repititions = 11
top_k = 5
stoploss = 1

overall_change = []

for repitition in range(repititions):

  try:
    stocks_parsed = [] #some stocks may run into errors, so this keeps tracks of the stocks that are parsed; keeps indexing corresponding to probas
    bull_proba = []
    bull_stocks = []
    bear_stocks = []
    bull_changes = [] #market neutral return calculation = long returns + short returns (assuming shorts have no cost, i.e. leveraging)
    bear_changes = []

    for i, stock in enumerate(stocks):
      print(f'[-----------------------{repitition+1, i/len(stocks)}------------------------]')
      try:
        verdict, proba = backtest_technical_model(stock, date)
        stocks_parsed.append(stock)
        bull_proba.append(proba[0][5]-proba[0][0])
      except Exception as e:
        print(stock, e)

    BP_TB = tie_break(bull_proba, 0.001)
    BP_TB_sorted = sorted(BP_TB)

    for i in range(top_k):
      bull_stocks.append(stocks_parsed[BP_TB.index(BP_TB_sorted[-(i+1)])]) #Bearest stocks are (most likely) negative, and are first after sorting
      bear_stocks.append(stocks_parsed[BP_TB.index(BP_TB_sorted[i])])

    for i in range(len(bull_stocks)):
      bull_changes.append(backtest_exit_rules(bull_stocks[i], date, stoploss=stoploss, action='long'))
      bear_changes.append(backtest_exit_rules(bear_stocks[i], date, stoploss=stoploss, action='short'))

    mo_change = (sum(bull_changes)/len(bull_changes)) + (sum(bear_changes)/len(bear_changes))
    overall_change.append(mo_change)

    print(f"Month's Change: {mo_change}")

    date = change_months(date, 1)

  except Exception as e:
    print(f'Repitition {repitition} Failed: {e}')
    date = change_months(date, 1)


Streaming output truncated to the last 5000 lines.
    🔹CDNS🔹[3]🔹🔷Bull🔷 🔹Bull Probability:0.2623662397597387
          
[-----------------------(6, 0.16458333333333333)------------------------]
[3] [[0.10131455 0.09533797 0.08869796 0.06029762 0.09271604 0.56163587]]

    🔹CZR🔹[3]🔹🔷Bull🔷 🔹Bull Probability:0.4603213199075189
          
[-----------------------(6, 0.16666666666666666)------------------------]
[2] [[0.20446751 0.10544418 0.11852805 0.19263607 0.26293366 0.11599053]]

    🔹CPT🔹[2]🔹◽?◽ 🔹Bull Probability:-0.08847697668928257
          
[-----------------------(6, 0.16875)------------------------]
[2] [[0.12045348 0.07249707 0.07212643 0.0743088  0.37953185 0.28108237]]

    🔹CPB🔹[2]🔹◽?◽ 🔹Bull Probability:0.16062889338193187
          
[-----------------------(6, 0.17083333333333334)------------------------]
[-1] [[0.15201737 0.18226128 0.19943578 0.14973096 0.12969231 0.18686229]]

    🔹COF🔹[-1]🔹◽?◽ 🔹Bull Probability:0.03484491487012753
          
[-----------------------(6,

KeyboardInterrupt: 

In [ ]:
date = "2026-04-13"


stocks_parsed=[]
stocks_bull_proba = []
stocks_bear_proba = []


weighted_bull_proba = []
weighted_bear_proba = []


E1_chosen_bull_stocks = []
E1_chosen_bear_stocks = []




for i in range(len(stocks)):
  try:
    print(f'- - - - - {((i+1)/len(stocks))*100}% Done - - - - -')
    verdict, proba = technical_model(stocks[i], date)


    stocks_bull_proba.append(proba[0][5])
    stocks_bear_proba.append(proba[0][0])
    stocks_parsed.append(stocks[i])


  except Exception:
    print(f"🔺🔺 Error processing {stocks[i]} 🔺🔺")


for i in range(len(stocks_bull_proba)):
  weighted_bull_proba.append(stocks_bull_proba[i]-stocks_bear_proba[i])
  weighted_bear_proba.append(stocks_bear_proba[i]-stocks_bull_proba[i])

tie_break(weighted_bull_proba, 0.001)
tie_break(weighted_bear_proba, 0.001)

weighted_bull_proba_sorted = sorted(weighted_bull_proba,reverse=True)
weighted_bear_proba_sorted = sorted(weighted_bear_proba,reverse=True)

tie_break(weighted_bull_proba_sorted, 0.001)
tie_break(weighted_bear_proba_sorted, 0.001)


for i in range(7):#############################get the most bull/bear stocks############################
  try:
    E1_chosen_bull_stocks.append(stocks_parsed[weighted_bull_proba.index(weighted_bull_proba_sorted[i])])
    E1_chosen_bear_stocks.append(stocks_parsed[weighted_bear_proba.index(weighted_bear_proba_sorted[i])])





  except Exception:
    print(f'''
    🚨 NOT ENOUGH STOCKS ARE CHOSEN 🚨
    {date}


    E1 Bull stocks: {E1_chosen_bull_stocks}
    E1 Bear stocks: {E1_chosen_bear_stocks}
    ''')


print(f'''


🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁


E1 Bull stocks:
{E1_chosen_bull_stocks}
Proba:
{weighted_bull_proba_sorted[:7]}


E1 Bear stocks:
{E1_chosen_bear_stocks}
Proba:
{weighted_bear_proba_sorted[:7]}


🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁




''')




- - - - - 0.20325203252032523% Done - - - - -
[-3] [[0.29902845 0.09917858 0.11540193 0.17024589 0.16879074 0.14735441]]
 🔹MMM🔹[-3]🔹🔷Bear🔷 🔹Bull Probability:0.14735441299199376          
- - - - - 0.40650406504065045% Done - - - - -
[-3] [[0.39015021 0.16383949 0.14070583 0.06820721 0.13748623 0.09961104]]
 🔹AOS🔹[-3]🔹🔷Bear🔷 🔹Bull Probability:0.09961103718735957          
- - - - - 0.6097560975609756% Done - - - - -
[3] [[0.13561785 0.19547832 0.12536256 0.12279591 0.07938699 0.34135839]]
 🔹ABT🔹[3]🔹🔷Bull🔷 🔹Bull Probability:0.3413583854309243          
- - - - - 0.8130081300813009% Done - - - - -
[1] [[0.09662656 0.08658729 0.13443499 0.41383007 0.14504553 0.12347556]]
 🔹ABBV🔹[1]🔹◽?◽ 🔹Bull Probability:0.12347556103026248          
- - - - - 1.0162601626016259% Done - - - - -
[3] [[0.14281324 0.07752186 0.05001844 0.10898985 0.11789274 0.50276387]]
 🔹ACN🔹[3]🔹🔷Bull🔷 🔹Bull Probability:0.5027638695688227          
- - - - - 1.2195121951219512% Done - - - - -
[3] [[0.27707843 0.13045946 0.090

In [ ]:
E1_50_bull_stocks = []
E1_50_bear_stocks = []

for i in range(20):#########################################################
    E1_50_bull_stocks.append(stocks_parsed[weighted_bull_proba.index(weighted_bull_proba_sorted[i])])
    E1_50_bear_stocks.append(stocks_parsed[weighted_bear_proba.index(weighted_bear_proba_sorted[i])])

print(f''' {date}
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦


Top 10 Bull:
{E1_50_bull_stocks[:10]}
Proba:
{weighted_bull_proba_sorted[:10]}

Top 15 Bull:
{E1_50_bull_stocks[:15]}
Proba:
{weighted_bull_proba_sorted[:15]}

🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦

Top 10 Bear:
{E1_50_bear_stocks[:10]}
Proba:
{weighted_bear_proba_sorted[:10]}

Top 15 Bear:
{E1_50_bear_stocks[:15]}
Proba:
{weighted_bear_proba_sorted[:15]}

🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
''')

 2026-02-14
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦


Top 10 Bull:
['MU', 'WDC', 'TER', 'DD', 'STLD', 'STX', 'CAT', 'FICO', 'LYB', 'PLTR']
Proba:
[np.float64(0.7182395155932826), np.float64(0.663311596632858), np.float64(0.647971525501058), np.float64(0.637829990396136), np.float64(0.6039639063489588), np.float64(0.6032766542064678), np.float64(0.5956129417950627), np.float64(0.5707690854613243), np.float64(0.5650262789331998), np.float64(0.5503925853820549)]

Top 15 Bull:
['MU', 'WDC', 'TER', 'DD', 'STLD', 'STX', 'CAT', 'FICO', 'LYB', 'PLTR', 'TPR', 'CZR', 'PANW', 'BLDR', 'SNPS']
Proba:
[np.float64(0.7182395155932826), np.float64(0.663311596632858), np.float64(0.647971525501058), np.float64(0.637829990396136), np.float64(0.6039639063489588), np.float64(0.6032766542064678), np.float64(0.5956129417950627), np.float64(0.5707690854613243), np.float64(0.5650262789331998), np.float64(

In [ ]:
stocks = [
    "NVDA", "GOOGL", "GOOG", "AAPL", "MSFT", "AMZN", "AVGO", "META", "TSLA", "WMT",
    "AMD", "ASML", "MU", "COST", "INTC", "NFLX", "CSCO", "PLTR", "LRCX", "AMAT",
    "KLAC", "TXN", "ARM", "LIN", "PEP", "TMUS", "ADI", "AMGN", "ISRG", "SHOP",
    "GILD", "QCOM", "APP", "PANW", "MRVL", "BKNG", "PDD", "WDC", "HON", "STX",
    "CRWD", "CEG", "SBUX", "INTU", "VRTX", "ADBE", "CMCSA", "MAR", "SNPS", "MELI",
    "CDNS", "ABNB", "CSX", "MPWR", "ADP", "ORLY", "DASH", "MNST", "REGN", "MDLZ",
    "AEP", "ROST", "CTAS", "BKR", "WBD", "PCAR", "FTNT", "NXP", "MSTR", "FANG",
    "FAST", "EA", "ADSK", "FER", "XEL", "MCHP", "EXC", "ODFL", "DDOG", "PYPL",
    "IDXX", "CCEP", "ALNY", "KDP", "TRI", "TTWO", "ROP", "PAYX", "AXON", "CPRT",
    "GEHC", "WDAY", "INSM", "CTSH", "KHC", "CHTR", "DXCM", "VRSK", "ZS", "TEAM",
    "CSGP"
]
len(stocks)

101

In [ ]:
stocks = ['^NDXT']

In [ ]:
for stock in stocks:
  time.sleep(0.25)
  ticker = yf.Ticker(stock).history(period = 'max')
  ticker.index = ticker.index.date
  ticker.to_excel(f'/content/drive/MyDrive/SP/NDXT data/{stock.upper()}.xlsx', index=True)
  print(stock, 'Done.')

^NDXT Done.


In [ ]:
error = []
for stock in stocks:
  ticker = yf.Ticker(stock).history(start = '2026-01-01', end='2026-04-26')
  if ticker.empty:
    error.append(stock)

for i in error:
  stocks.remove(i)

print(stocks)

['NVDA', 'GOOGL', 'GOOG', 'AAPL', 'MSFT', 'AMZN', 'AVGO', 'META', 'TSLA', 'WMT', 'AMD', 'ASML', 'MU', 'COST', 'INTC', 'NFLX', 'CSCO', 'PLTR', 'LRCX', 'AMAT', 'KLAC', 'TXN', 'ARM', 'LIN', 'PEP', 'TMUS', 'ADI', 'AMGN', 'ISRG', 'SHOP', 'GILD', 'QCOM', 'APP', 'PANW', 'MRVL', 'BKNG', 'PDD', 'WDC', 'HON', 'STX', 'CRWD', 'CEG', 'SBUX', 'INTU', 'VRTX', 'ADBE', 'CMCSA', 'MAR', 'SNPS', 'MELI', 'CDNS', 'ABNB', 'CSX', 'MPWR', 'ADP', 'ORLY', 'DASH', 'MNST', 'REGN', 'MDLZ', 'AEP', 'ROST', 'CTAS', 'BKR', 'WBD', 'PCAR', 'FTNT', 'NXP', 'MSTR', 'FANG', 'FAST', 'EA', 'ADSK', 'FER', 'XEL', 'MCHP', 'EXC', 'ODFL', 'DDOG', 'PYPL', 'IDXX', 'CCEP', 'ALNY', 'KDP', 'TRI', 'TTWO', 'ROP', 'PAYX', 'AXON', 'CPRT', 'GEHC', 'WDAY', 'INSM', 'CTSH', 'KHC', 'CHTR', 'DXCM', 'VRSK', 'ZS', 'TEAM', 'CSGP']


In [ ]:
error

['DAY', 'MMC']

In [ ]:
import tpot
from tpot import TPOTClassifier

In [ ]:
stock = "aapl"
date = "2026-01-01"
X,y = data(stock, change_months(date, -72), date,shuffle = False)

X_test = X.iloc[-200:]
y_test = y.iloc[-200:]

X = X.iloc[:-222]
y = y.iloc[:-222]

X = pd.concat([X,y], axis=1)
X_train = X.sample(frac=1, random_state=1)

y_train = X_train.pop("label")

"Train: ", X_train.shape, y_train.shape, "Test:", X_test.shape, y_train.shape

('Train: ', (1197, 289), (1197,), 'Test:', (200, 289), (1197,))

In [ ]:
X_train.shape

(1197, 289)

In [ ]:
TPOT = tpot.TPOTClassifier(
    #scoring = 'accuracy',
    generations = 10, #how many interations per algorithm is tested
    population_size = 20,
    verbose=1,
    cv = 5,
    max_time_mins = 60
    )
TPOT.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/tpot/tpot_estimator/estimator.py:458: UserWarning: Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.
  warnings.warn("Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.")
INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:35749
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:34769'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:39433 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:39433
INFO:distributed.core:Starting established connection to tcp://127.0.0.1

TPOTClassifier(cv=5,
               search_space=<tpot.search_spaces.pipelines.sequential.SequentialPipeline object at 0x78d414223f50>,
               verbose=1)

In [ ]:
score = TPOT.fitted_pipeline_.score(X_test, y_test)
print(f"Test Score: {score}")
# Print the entire pipeline structure
print(TPOT.fitted_pipeline_)

# To specifically see the hyperparameters of the final model:
best_step_name = list(TPOT.fitted_pipeline_.named_steps.keys())
print(f"Model: {best_step_name}")
print(f"Hyperparameters: {TPOT.fitted_pipeline_.named_steps[best_step_name].get_params()}")


Test Score: 0.0
Pipeline(steps=[('maxabsscaler', MaxAbsScaler()),
                ('selectfwe', SelectFwe(alpha=0.0003078185684)),
                ('featureunion-1',
                 FeatureUnion(transformer_list=[('skiptransformer',
                                                 SkipTransformer()),
                                                ('passthrough',
                                                 Passthrough())])),
                ('featureunion-2',
                 FeatureUnion(transformer_list=[('skiptransformer',
                                                 SkipTransformer()),
                                                ('passthrough',
                                                 Passthrough())])),
                ('baggingclassifier',
                 BaggingClassifier(bootstrap=np.True_,
                                   bootstrap_features=np.False_,
                                   max_features=0.6524941957496,
                                   max

In [ ]:
BG = BaggingClassifier(
    bootstrap=np.True_,
    bootstrap_features=np.False_,
    estimator=None,
    max_features=0.6524941957496,
    max_samples=0.8359000837192,
    n_estimators=100,
    n_jobs=1,
    oob_score=np.False_,
    random_state=None,
    verbose=0,
    warm_start=False
)


In [ ]:
BG.fit(X_train, y_train)

BaggingClassifier(bootstrap=np.True_, bootstrap_features=np.False_,
                  max_features=0.6524941957496, max_samples=0.8359000837192,
                  n_estimators=100, n_jobs=1, oob_score=np.False_)

In [ ]:
BG.score(X_test, y_test)

0.17

In [ ]:
vote.fit(X_train, y_train)

VotingClassifier(estimators=[('ADARF',
                              AdaBoostClassifier(estimator=RandomForestClassifier(random_state=1),
                                                 random_state=1)),
                             ('RF', RandomForestClassifier(random_state=1)),
                             ('LGBM',
                              LGBMClassifier(random_state=1, verbose=-1)),
                             ('ET', ExtraTreesClassifier(random_state=1))],
                 voting='soft')

In [ ]:
best_step_name = list(TPOT.fitted_pipeline_.named_steps.keys())
print(best_step_name)

['maxabsscaler', 'selectfwe', 'featureunion-1', 'featureunion-2', 'baggingclassifier']


In [ ]:
print(f"Hyperparameters: {TPOT.fitted_pipeline_.named_steps[best_step_name[2]].get_params()}")


Hyperparameters: {'n_jobs': None, 'transformer_list': [('skiptransformer', SkipTransformer()), ('passthrough', Passthrough())], 'transformer_weights': None, 'verbose': False, 'verbose_feature_names_out': True, 'skiptransformer': SkipTransformer(), 'passthrough': Passthrough()}


In [ ]:
#model name = BG

#Hyperparameters: Maxbsscaler {copy=True}
#Selectfwe {'alpha': 0.0003078185684, 'score_func': <function f_classif at 0x78d44cd9c4a0>}
#featureunion-1 Hyperparameters: {'n_jobs': None, 'transformer_list': [('skiptransformer', SkipTransformer()), ('passthrough', Passthrough())], 'transformer_weights': None, 'verbose': False, 'verbose_feature_names_out': True, 'skiptransformer': SkipTransformer(), 'passthrough': Passthrough()}
#featureunion-2 Hyperparameters: {'n_jobs': None, 'transformer_list': [('skiptransformer', SkipTransformer()), ('passthrough', Passthrough())], 'transformer_weights': None, 'verbose': False, 'verbose_feature_names_out': True, 'skiptransformer': SkipTransformer(), 'passthrough': Passthrough()}


In [ ]:
import numpy as np
from sklearn.preprocessing import MaxAbsScaler
from sklearn.feature_selection import SelectFwe, f_classif
from sklearn.pipeline import FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin

# Assume X_train, y_train, X_test, y_test are already defined

# Step 1: Apply MaxAbsScaler with copy=True
scaler = MaxAbsScaler(copy=True)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 2: SelectFwe with given parameters
select_fwe = SelectFwe(alpha=0.0003078185684, score_func=f_classif)
X_train_selected = select_fwe.fit_transform(X_train_scaled, y_train)
X_test_selected = select_fwe.transform(X_test_scaled)

# Step 3: Create FeatureUnion-1
feature_union_1 = FeatureUnion(
    transformer_list=[
        ('original', 'passthrough'),  # Keeps one copy of original data
        ('scaled', StandardScaler())  # Adds a scaled copy of the data
    ],
    n_jobs=None,
    transformer_weights=None,
    verbose=False,
    verbose_feature_names_out=True
)

# Fit and transform train data, transform test data for FeatureUnion-1
X_train_fu1 = feature_union_1.fit_transform(X_train_selected, y_train)
X_test_fu1 = feature_union_1.transform(X_test_selected)

# Step 4: Create FeatureUnion-2 (same settings)
feature_union_2 = FeatureUnion(
    transformer_list=[
        ('original', 'passthrough'),  # Keeps one copy of original data
        ('scaled', StandardScaler())  # Adds a scaled copy of the data
    ],
    n_jobs=None,
    transformer_weights=None,
    verbose=False,
    verbose_feature_names_out=True
)

X_train_processed = feature_union_2.fit_transform(X_train_fu1, y_train)
X_test_processed = feature_union_2.transform(X_test_fu1)



/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [5] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [ ]:
BG.fit(X_train_processed, y_train)

BaggingClassifier(bootstrap=np.True_, bootstrap_features=np.False_,
                  max_features=0.6524941957496, max_samples=0.8359000837192,
                  n_estimators=100, n_jobs=1, oob_score=np.False_)

In [ ]:
ypred = BG.predict(X_test_processed)
cm = confusion_matrix(y_test, ypred)
cf = classification_report(y_test, ypred)

print(cm,"\n",cf)

[[ 0  0  0  0  0  0]
 [ 0  0  3  1  0  0]
 [ 1  0 38 13  0  6]
 [ 6  0 70 30  0  3]
 [ 0  0  5  1  0  0]
 [ 0  0 17  6  0  0]] 
               precision    recall  f1-score   support

          -3       0.00      0.00      0.00         0
          -2       0.00      0.00      0.00         4
          -1       0.29      0.66      0.40        58
           1       0.59      0.28      0.38       109
           2       0.00      0.00      0.00         6
           3       0.00      0.00      0.00        23

    accuracy                           0.34       200
   macro avg       0.15      0.16      0.13       200
weighted avg       0.40      0.34      0.32       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_